# CURRENT_NLR_PBE_GW_V1 descriptor/reference-graph gate

Use a standard Colab CPU runtime and choose **Runtime -> Run all**. This notebook clones and pins GitHub `main`, creates an isolated Python 3.12 virtual environment, installs the fully locked graph dependencies, and runs only the descriptor/reference-graph compatibility gate. It does not construct preference factors, perform influence calculations, run inference, or run Bayesian optimization. GW oracle data are never opened.

In [ ]:
import csv
import hashlib
import importlib.metadata as md
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

REPOSITORY_URL = 'https://github.com/PaulsonLab/energy-inference-bo.git'
REPOSITORY_DIR = Path('/content/energy-inference-bo')
if REPOSITORY_DIR.exists():
    shutil.rmtree(REPOSITORY_DIR)
subprocess.run(['git', 'clone', '--branch', 'main', '--single-branch', REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
RUN_SHA = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], cwd=REPOSITORY_DIR, check=True, capture_output=True, text=True
).stdout.strip()
subprocess.run(['git', 'checkout', '--detach', RUN_SHA], cwd=REPOSITORY_DIR, check=True)
assert subprocess.run(
    ['git', 'rev-parse', 'HEAD'], cwd=REPOSITORY_DIR, check=True, capture_output=True, text=True
).stdout.strip() == RUN_SHA
print('RUN_SHA', RUN_SHA)

In [ ]:
BENCHMARK_DIR = REPOSITORY_DIR / 'experiments/sun_oxide/benchmark'
MANIFEST_PATH = BENCHMARK_DIR / 'benchmark_manifest.json'
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
assert manifest['benchmark_name'] == 'CURRENT_NLR_PBE_GW_V1'
assert manifest['counts']['legacy_compositions'] == 2142
assert manifest['counts']['strict_gw_actions'] == 191
assert manifest['target_isolation']['gw_target_used_for_candidate_selection'] is False
assert manifest['target_isolation']['gw_target_used_for_strict_mapping'] is False
safe_artifacts = [
    'current_nlr_legacy.csv',
    'current_nlr_gw_actions.csv',
    'NLR_DATA_USE_NOTICE.txt',
]
for name in safe_artifacts:
    path = BENCHMARK_DIR / name
    observed = hashlib.sha256(path.read_bytes()).hexdigest()
    assert observed == manifest['artifacts'][name]['sha256'], name
def csv_data_rows(path):
    with path.open('r', encoding='utf-8', newline='') as stream:
        return sum(1 for _ in csv.reader(stream)) - 1
assert csv_data_rows(BENCHMARK_DIR / 'current_nlr_legacy.csv') == 2142
assert csv_data_rows(BENCHMARK_DIR / 'current_nlr_gw_actions.csv') == 191
print('FROZEN_BENCHMARK_VERIFIED legacy=2142 actions=191 gw_values_read=false')

In [ ]:
VENV_DIR = Path('/content/sunoxide_graph_venv')
UV_PYTHON_DIR = Path('/content/sunoxide_uv_python')
LOCK_PATH = REPOSITORY_DIR / 'experiments/sun_oxide/requirements-colab-graph.txt'
UV_BOOTSTRAP_VERSION = '0.10.11'
PYTHON_VERSION = '3.12.13'
required_versions = {
    'matminer': '0.10.1',
    'numpy': '2.3.5',
    'pandas': '2.3.3',
    'scipy': '1.18.0',
    'scikit-learn': '1.9.0',
}
try:
    if VENV_DIR.exists():
        shutil.rmtree(VENV_DIR)
    if UV_PYTHON_DIR.exists():
        shutil.rmtree(UV_PYTHON_DIR)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
        f'uv=={UV_BOOTSTRAP_VERSION}',
    ], check=True)
    uv_environment = os.environ.copy()
    uv_environment['UV_PYTHON_INSTALL_DIR'] = str(UV_PYTHON_DIR)
    subprocess.run([
        sys.executable, '-m', 'uv', 'python', 'install', PYTHON_VERSION,
        '--install-dir', str(UV_PYTHON_DIR), '--no-bin',
    ], check=True, env=uv_environment)
    subprocess.run([
        sys.executable, '-m', 'uv', 'venv', '--python', PYTHON_VERSION,
        '--managed-python', str(VENV_DIR),
    ], check=True, env=uv_environment)
    VENV_PYTHON = VENV_DIR / 'bin/python'
    subprocess.run([str(VENV_PYTHON), '-m', 'ensurepip', '--upgrade'], check=True)
    subprocess.run([
        str(VENV_PYTHON), '-m', 'pip', 'install', '--require-hashes', '--no-deps', '-r', str(LOCK_PATH)
    ], check=True)
    subprocess.run([str(VENV_PYTHON), '-m', 'pip', 'check'], check=True)
    smoke_code = r'''
import importlib.metadata as md
import json
import math
import sys
from matminer.featurizers.composition import ElementProperty
from pymatgen.core import Composition
required = {'matminer':'0.10.1','numpy':'2.3.5','pandas':'2.3.3','scipy':'1.18.0','scikit-learn':'1.9.0'}
observed = {name: md.version(name) for name in required}
assert sys.version_info[:2] == (3, 12), sys.version
assert observed == required, (observed, required)
featurizer = ElementProperty.from_preset('magpie', impute_nan=True)
labels = featurizer.feature_labels()
assert len(labels) == 132, len(labels)
for formula in ('Ag Al O2', 'O7 Y2 Zr2'):
    values = featurizer.featurize(Composition(formula))
    assert len(values) == 132
    assert all(math.isfinite(float(value)) for value in values)
key = required | {name: md.version(name) for name in ('monty','pymatgen','pymatgen-core','sympy')}
print(json.dumps({'python':sys.version.split()[0], 'versions':key, 'feature_count':len(labels), 'smoke_formulas':2}, sort_keys=True))
'''
    environment_check = subprocess.run(
        [str(VENV_PYTHON), '-c', smoke_code], check=True, capture_output=True, text=True
    )
    print(environment_check.stdout.strip())
except Exception as exc:
    print('INSTALLATION_BLOCKED', str(exc))
    raise

In [ ]:
OUTPUT_DIR = Path('/content/sun_oxide_descriptor_graph_outputs')
ZIP_PATH = Path('/content/sun_oxide_descriptor_graph_outputs.zip')
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
command = [
    str(VENV_PYTHON),
    str(REPOSITORY_DIR / 'experiments/sun_oxide/descriptor_graph.py'),
    '--config', str(REPOSITORY_DIR / 'experiments/sun_oxide/configs/descriptor_graph.json'),
    '--repository-root', str(REPOSITORY_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--run-sha', RUN_SHA,
    '--zip-path', str(ZIP_PATH),
]
completed = subprocess.run(command, cwd=REPOSITORY_DIR)
if completed.returncode != 0:
    raise RuntimeError('IMPLEMENTATION_BLOCKED: descriptor_graph.py did not complete; see its terminal verdict above')

In [ ]:
summary = json.loads((OUTPUT_DIR / 'run_summary.json').read_text(encoding='utf-8'))
artifact_manifest = json.loads((OUTPUT_DIR / 'artifact_manifest.json').read_text(encoding='utf-8'))
assert summary['verdict'] == 'PASS_DESCRIPTOR_GRAPH_COLAB'
assert summary['run_sha'] == RUN_SHA
assert summary['descriptor']['shape'] == [2142, 132]
assert summary['descriptor']['nonfinite_count'] == 0
assert summary['graph']['connected_component_count'] == 1
assert summary['graph']['isolated_node_count'] == 0
assert summary['actions']['mapped_actions'] == 191
assert summary['target_isolation']['gw_values_read'] is False
assert artifact_manifest['run_sha'] == RUN_SHA
assert len(artifact_manifest['files']) == 9
for entry in artifact_manifest['files']:
    path = OUTPUT_DIR / entry['path']
    assert path.stat().st_size == entry['size_bytes']
    assert hashlib.sha256(path.read_bytes()).hexdigest() == entry['sha256']
assert ZIP_PATH.is_file()
with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
    assert archive.namelist() == [entry['path'] for entry in artifact_manifest['files']] + ['artifact_manifest.json']
zip_sha256 = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
print('RUN_SHA', RUN_SHA)
print('ZIP_PATH', ZIP_PATH)
print('ZIP_SHA256', zip_sha256)
print(json.dumps({
    'descriptor_shape': summary['descriptor']['shape'],
    'zero_variance_features': summary['descriptor']['zero_variance_feature_count'],
    'knn_edges': summary['graph']['knn_edge_count'],
    'mst_edges': summary['graph']['mst_edge_count'],
    'final_edges': summary['graph']['final_unique_edge_count'],
    'connected_components': summary['graph']['connected_component_count'],
    'mapped_actions': summary['actions']['mapped_actions'],
}, sort_keys=True))
print('PASS_DESCRIPTOR_GRAPH_COLAB')